# REDBOX-714: Contact details for user research

## Background

Katie’s guidance about the different variables/ sub groups:

So the two main things to select users by are:

* Regular users of Redbox
* Using documents and or multiple sources (chat with and without documents)

Then within those, we need representation of:

* Different business units
* Different professions
* Women (we had no women in the recent round)

## Load packages

In [1]:
import numpy as np
import pandas as pd
import datetime
import plotly.express as px
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import tkinter as tk
from tkinter import filedialog

## Read in data from Redbox app

In [2]:
def select_file():
    root = tk.Tk()
    root.withdraw() 
    file_path = filedialog.askopenfilename() 
    return file_path

### Chat data

In [3]:
# Load the chat table
chat_file = select_file()
chat_raw = (
    pd.read_csv(chat_file)
    .sort_values(by="message_created_at", ascending=False)
    .reset_index(drop=True)
)


In [ ]:
# Flag i.AI users in data
iai_file = select_file()
iai_users = pd.read_csv(iai_file, index_col=False)
chat_raw["is_iai_user"] = np.isin(chat_raw["history_user"], iai_users["user_email"])
chat = chat_raw.loc[chat_raw['is_iai_user'] == False]


In [ ]:
# Format timestamps and create variables useful for plotting
chat["message_created_at"] = pd.to_datetime(chat["message_created_at"])
chat["message_date"] = chat["message_created_at"].dt.date
chat['message_created_week'] = chat['message_created_at'].dt.to_period('W').dt.to_timestamp()
chat["message_text"] = chat["message_text"].fillna("NULL")
chat['min_message_created_at'] = chat.groupby('history_user')['message_created_at'].transform('min')
chat['delta_since_first_message'] = chat['message_created_at'] - chat['min_message_created_at']
chat['week_of_message'] = np.floor(chat['delta_since_first_message']/np.timedelta64(1, 'W')) + 1

chat["history_created_at"] = pd.to_datetime(chat["history_created_at"])
chat["history_date"] = chat["history_created_at"].dt.date
chat['min_history_created_at'] = chat.groupby('history_user')['history_created_at'].transform('min')
chat['delta_since_first_history'] = chat['history_created_at'] - chat['min_history_created_at']
chat['week_of_conversation'] = np.floor(chat['delta_since_first_history']/np.timedelta64(1, 'W')) + 1

chat['delta_first_message_to_present'] = datetime.datetime.now(datetime.timezone.utc) - chat['min_message_created_at']
chat['days_first_message_to_present'] = np.floor(chat['delta_first_message_to_present']/np.timedelta64(1, 'D')) + 1
chat['rating_provided'] = np.where(chat['message_rating'].isna(), False, True)

### User-level data

In [ ]:
# Load the user table
user_file = select_file()
user = (
    pd.read_csv(user_file)
)

### Join it together

In [ ]:
joined = pd.merge(
    left=chat,
    right=user,
    left_on='history_user',
    right_on='email',
    validate='many_to_one'
    )

## Create user-level usage table

In [ ]:
# Filter to user messages
user_messages = joined[joined['message_role'] == "user"]

# One-hot encode the message_route variable so that we want can create user-level proportions later
one_hotted = pd.get_dummies(
    joined,
    prefix='route',
    columns=['message_route']
)

characteristics_per_user = (
    one_hotted
    .groupby('email', as_index = False)
    .agg(
        mean_message_rating=('message_rating', 'mean'),
        median_message_rating=('message_rating', 'median'),
        prop_rating_provided=('rating_provided', 'mean'),
        business_unit=('business_unit', 'first'),
        grade=('grade', 'first'),
        ai_experience=('ai_experience', 'first'),
        profession=('profession', 'first'),
        consent_research=('consent_research', 'first'),
        consent_interviews=('consent_interviews', 'first'),
        consent_feedback=('consent_feedback', 'first'),
        n_days_since_first_message=('days_first_message_to_present', 'max'),
        prop_route_chat=('route_chat', 'mean'),
        prop_route_chat_docs=('route_chat/documents', 'mean'),
        prop_route_chat_docs_large=('route_chat/documents/large', 'mean'),
        prop_route_search=('route_search', 'mean')
        )
    )

characteristics_per_user['ai_experience'] = characteristics_per_user['ai_experience'].astype('category')
characteristics_per_user['ai_experience'] = characteristics_per_user['ai_experience'].cat.reorder_categories(["Curious Newcomer", "Cautious Explorer", "Enthusiastic Experimenter", "Experienced Navigator", "AI Alchemist"])

usage_per_user = (
    user_messages
    .groupby('email', as_index = False)
    .agg(
        n_conversations=('history_id', 'nunique'),
        n_messages=('message_id', 'nunique'),
        n_days_since_first_message=('days_first_message_to_present', 'max')
        )
    )

usage_per_user['n_messages_per_day'] = usage_per_user['n_messages']/usage_per_user['n_days_since_first_message']
usage_per_user['n_conversations_per_day'] = usage_per_user['n_conversations']/usage_per_user['n_days_since_first_message']
usage_per_user = usage_per_user.drop(['n_days_since_first_message'], axis=1)

per_user_df = pd.merge(
    left=usage_per_user,
    right=characteristics_per_user,
    left_on='email',
    right_on='email',
    validate='one_to_one'
    )

#TODO: Why are we losing users?

## Find regular users who use documents

In [ ]:
# Get descriptive stats for usage so that we can pick thresholds
per_user_df['n_conversations_per_day'].describe()

# If we said 'regular users' were people in certain part of both the distributions, then how many would we have?
usage_lower, usage_upper = per_user_df['n_conversations_per_day'].quantile([0.75, 1])
doc_lower, doc_upper = per_user_df['prop_route_chat_docs'].quantile([0.75, 1])

condition_1 = per_user_df['n_conversations_per_day'] > usage_lower
condition_2 = per_user_df['n_conversations_per_day'] < usage_upper
condition_3 = per_user_df['prop_route_chat_docs'] > doc_lower
condition_4 = per_user_df['prop_route_chat_docs'] < doc_upper
condition_5 = per_user_df['consent_interviews'] == 1

filtered_per_user_df = per_user_df[condition_1 & condition_2 & condition_3 & condition_4 & condition_5].copy()

# Replace Null values so that we can get stratified sample among professions
filtered_per_user_df['business_unit'] = filtered_per_user_df['business_unit'].fillna('missing')
filtered_per_user_df['profession'] = filtered_per_user_df['profession'].fillna('missing')

# Find the business units we have in the data
business_units = filtered_per_user_df['business_unit'].unique()

# Convert to dictionary to define how many samples we'll take within each business unit
business_units_dict = {unit: 2 for unit in business_units}
business_units_dict['missing'] = 10
business_units_dict


In [ ]:
stratified_df = (
    filtered_per_user_df
    .groupby('business_unit', group_keys=False)
    .apply(lambda x: x.sample(n=business_units_dict[x.name], replace=True, random_state = 1))
)

stratified_df_dedup = stratified_df.drop_duplicates(keep='first')


# Write out the people we want to contact to a table

In [ ]:
to_contact = stratified_df_dedup[
    ['email',
     'n_conversations',
     'n_conversations_per_day',
     'n_messages',
     'n_messages_per_day',
     'prop_route_chat',
     'prop_route_chat_docs',
     'business_unit',
     'grade',
     'ai_experience',
     'profession',
     'consent_interviews']
     ].round(
         {'n_conversations_per_day': 2,
          'n_messages_per_day': 2,
          'prop_route_chat': 3,
          'prop_route_chat_docs': 3}
          )